# Optuna HPO trên Kaggle — CIES Fraud Detection

Notebook độc lập, import thẳng vào Kaggle (Notebook -> File -> Import Notebook), không phụ thuộc file nào khác ngoài repo GitHub.

## Trước khi chạy
1. Bật **Settings -> Internet: On** (cần để `git clone` + `pip install`).
2. (Tuỳ chọn) Bật **Accelerator: GPU** nếu muốn tune `xgboost`/`catboost`/`ann` nhanh hơn — cả 3 đã tự nhận GPU nếu có.
3. Repo phải ở chế độ **Public** trên GitHub (Settings -> Danger Zone -> Change visibility) để Kaggle clone ẩn danh được.
4. **Add Input** 1 trong 2 cách lấy dữ liệu đã tiền xử lý:
   - **Cách A (khuyến nghị, nhanh nhất)**: upload `data/processed/train_encoded.parquet` (và `test_encoded.parquet` nếu cần) từ máy local lên thành 1 Kaggle Private Dataset (ví dụ tên `cies-processed`), rồi Add Input dataset đó vào notebook này. Bỏ qua bước tiền xử lý.
   - **Cách B**: Add Input dataset gốc `kartik2112/fraud-detection` (Sparkov) và tự chạy lại `notebooks/02_preprocessing.ipynb` trên Kaggle để tạo ra `train_encoded.parquet` trước, rồi mới chạy notebook này.
5. Sửa `MODELS_TO_TUNE` ở cell config bên dưới nếu muốn tune model khác/nhiều hơn 2 model.


## 1. Clone code + cài dependencies

In [ ]:
import os

REPO_URL = "https://github.com/gnUrt1106/fraud-detection-CIES.git"
REPO_DIR = "/kaggle/working/fraud-detection-CIES"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


In [ ]:
# Kaggle image đã có sẵn hầu hết (pandas, sklearn, xgboost, torch...).
# Chỉ cần đảm bảo optuna + catboost + pyarrow đúng version.
# PHẢI đặt spec trong ngoặc kép: không có thì shell hiểu ">" là chuyển hướng
# stdout ra file tên "=3.4.0", "=1.2"... (vừa bỏ mất ràng buộc version, vừa để
# lại file rác trong thư mục output).
!pip install -q "optuna>=3.4.0" "catboost>=1.2" "pyarrow>=14.0.0"

## 2. Chuẩn bị dữ liệu (Cách A — dùng parquet đã xử lý sẵn)

Tự động tìm `train_encoded.parquet` trong `/kaggle/input/**` và symlink vào `data/processed/`. Nếu bạn đi theo **Cách B**, hãy tự chạy `notebooks/02_preprocessing.ipynb` trước rồi bỏ qua cell này.

In [ ]:
import glob
import os

from src.config import PROCESSED_DATA_DIR

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

candidates = glob.glob("/kaggle/input/**/train_encoded.parquet", recursive=True)
if candidates:
    src_path = candidates[0]
    dst_path = PROCESSED_DATA_DIR / "train_encoded.parquet"
    if not dst_path.exists():
        os.symlink(src_path, dst_path)
    print(f"Đã liên kết: {src_path} -> {dst_path}")
else:
    print(
        "KHÔNG tìm thấy train_encoded.parquet trong /kaggle/input.\n"
        "-> Nếu đi Cách A: kiểm tra lại đã Add Input đúng dataset chứa file này chưa.\n"
        "-> Nếu đi Cách B: chạy notebooks/02_preprocessing.ipynb trước để tạo file này "
        "trong data/processed/, rồi chạy lại từ cell này."
    )

## 3. Config — chọn model cần tune

In [ ]:
# Sửa danh sách này để tune model khác. Các giá trị hợp lệ:
# logistic_regression, random_forest, xgboost, catboost, ann
# 4 model đầu đã tune xong (xem results/best_params.json) — chỉ còn ann.
MODELS_TO_TUNE = ["ann"]

N_TRIALS = 30
N_SPLITS = 5


## 4. Load dữ liệu + chạy Optuna HPO

Dùng `tune_all_models` gốc (qua `run_isolated`/`multiprocessing spawn`) — trên Linux của Kaggle cách này chạy ổn định, không bị deadlock như trên macOS.

In [ ]:
import pandas as pd

from src.config import PROCESSED_DATA_DIR, TARGET_COL

train_df = pd.read_parquet(PROCESSED_DATA_DIR / "train_encoded.parquet")
X = train_df.drop(columns=[TARGET_COL]).values
y = train_df[TARGET_COL].values

print(f"Train shape: {X.shape}, fraud rate: {y.mean():.4%}")

In [ ]:
from src.models.tune import tune_all_models

all_results = tune_all_models(
    X, y,
    models=MODELS_TO_TUNE,
    n_trials=N_TRIALS,
    n_splits=N_SPLITS,
)

## 5. Kết quả

`results/best_params.json` được ghi trong `/kaggle/working/fraud-detection-CIES/results/` — Kaggle tự giữ lại các file trong `/kaggle/working` khi bạn **Save Version**, tải về qua tab Output.

In [ ]:
import json

from src.config import RESULTS_DIR

for model_name, res in all_results.items():
    print(f"{model_name}: best PR-AUC = {res['best_pr_auc']:.4f}")
    print(f"  best_params = {res['best_params']}\n")

print("File kết quả:", RESULTS_DIR / "best_params.json")